# Attention Head Classification

Classify attention heads by functional role: induction, previous-token,
positional, copy, and combined summary.

## Why This Matters

Attention head classification reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_classification import (
    detect_induction_heads, detect_previous_token_heads,
    detect_positional_heads, detect_copy_heads,
    head_classification_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
# Use repeated tokens to enable induction detection
tokens = jnp.array([1, 15, 30, 45, 1, 15, 30])
print('Model ready')

## Induction Heads

Induction heads attend to the token after a previous occurrence of the current context.

In [ ]:
result = detect_induction_heads(model, tokens)
print(f"Induction heads found: {result['n_induction_heads']}\n")
for h in result['heads'][:8]:
    flag = ' ◄ INDUCTION' if h['is_induction'] else ''
    print(f"  L{h['layer']}H{h['head']}: score={h['induction_score']:.4f}, "
          f"matches={h['match_count']}{flag}")

## Previous Token Heads

Heads that attend primarily to the immediately preceding token.

In [ ]:
result = detect_previous_token_heads(model, tokens)
print(f"Previous-token heads found: {result['n_previous_token_heads']}\n")
for h in result['heads'][:8]:
    bar = '█' * int(h['prev_token_score'] * 30)
    flag = ' ◄ PREV' if h['is_previous_token'] else ''
    print(f"  L{h['layer']}H{h['head']}: score={h['prev_token_score']:.4f} {bar}{flag}")

## Positional Heads

Heads with fixed positional attention patterns (BOS or self-attention).

In [ ]:
result = detect_positional_heads(model, tokens)
print(f"Positional heads found: {result['n_positional_heads']}\n")
for h in result['heads'][:8]:
    flag = ' ◄ POS' if h['is_positional'] else ''
    print(f"  L{h['layer']}H{h['head']}: bos={h['bos_attention']:.4f}, "
          f"self={h['self_attention']:.4f}, entropy={h['mean_entropy']:.4f}{flag}")

## Copy Heads

Heads whose OV circuit maps token embeddings to similar output directions.

In [ ]:
result = detect_copy_heads(model, tokens)
print(f"Copy heads found: {result['n_copy_heads']}\n")
for h in result['heads'][:8]:
    flag = ' ◄ COPY' if h['is_copy'] else ''
    print(f"  L{h['layer']}H{h['head']}: mean={h['mean_copy_score']:.4f}, "
          f"max={h['max_copy_score']:.4f}{flag}")

## Classification Summary

Combined classification of all heads into functional categories.

In [ ]:
result = head_classification_summary(model, tokens)
print(f"Induction: {result['n_induction']}, Prev-token: {result['n_previous_token']}, "
      f"Positional: {result['n_positional']}, Copy: {result['n_copy']}, "
      f"Other: {result['n_other']}\n")
for c in result['classifications']:
    cats = ', '.join(c['categories'])
    print(f"  L{c['layer']}H{c['head']}: {cats}")